# Smartphone Addiction: Baseline Modeling

Playground Series S6E8 — sanity baselines, strong native-categorical models, an `_is_missing` out-of-fold ablation, engineered ratio/residual features, and a class-imbalance A/B, each tested directly on cross-validated AUC rather than assumed. Two modes: `evaluate` runs the full comparison; `submission` fits the selected champion on all training data and writes a submission file.

## 1. Config

In [ ]:
import os
import time
from typing import Literal

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

RUN_MODE: Literal["evaluate", "submission"] = "evaluate"
CHAMPION_NAME = "lightgbm_tuned"
NOTEBOOK_VERSION = "e01-lightgbm-v1"  # Task 7 freeze: final champion identity

if RUN_MODE not in {"evaluate", "submission"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

# Mode flags: one per experiment block, so each can be toggled without
# commenting code in/out. All evaluation-only work is skipped in submission
# mode, which goes straight to load -> infer -> write.
RUN_V1_SANITY = RUN_MODE == "evaluate"
RUN_LOGISTIC = False  # retired: still numerically unstable even with
                       # solver="saga", penalty="l2", C=0.1, max_iter=2_000
RUN_V2_STRONG = RUN_MODE == "evaluate"
RUN_V2_MISSING_ABLATION = RUN_MODE == "evaluate"
RUN_V3_ENGINEERED = RUN_MODE == "evaluate"
RUN_CLASS_WEIGHT_ABLATION = RUN_MODE == "evaluate"
RUN_E01_TUNING = RUN_MODE == "evaluate"
RUN_E02_ENTRY_CHECK = RUN_MODE == "evaluate"
RUN_E03_EXPLORATION = RUN_MODE == "evaluate"
RUN_SUMMARY = RUN_MODE == "evaluate"

pd.set_option("display.max_columns", 50)

## 2. Data Loading

In [ ]:
if os.path.exists("/kaggle/input/competitions/playground-series-s6e8"):
    DATA_DIR = "/kaggle/input/competitions/playground-series-s6e8"
else:
    DATA_DIR = "../data"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

TARGET = "addicted_label"
NUMERIC_FEATURES = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CATEGORICAL_FEATURES = ["gender", "stress_level", "academic_work_impact"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = train[ALL_FEATURES].copy()
y = train[TARGET].copy()
X_test = test[ALL_FEATURES].copy()

for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

## 3. Cross-Validation Helper

In [ ]:
results = []  # (name, oof_auc, fold_aucs) for the summary table
oof_store = {}  # name -> oof predictions, for sanity checks / future ensembling

def run_cv(name: str, fit_predict_fold, X_df: pd.DataFrame, y_ser: pd.Series) -> np.ndarray:
    """Run stratified 5-fold CV, print per-fold and overall OOF AUC.

    Args:
        name: label for the results table.
        fit_predict_fold: callable(X_tr, y_tr, X_val) -> val_pred_proba.
        X_df: feature frame.
        y_ser: target series.

    Returns:
        OOF prediction array aligned to X_df's row order.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_df))
    fold_aucs = []
    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_df, y_ser)):
        X_tr, X_val = X_df.iloc[tr_idx], X_df.iloc[val_idx]
        y_tr, y_val = y_ser.iloc[tr_idx], y_ser.iloc[val_idx]
        val_pred = fit_predict_fold(X_tr, y_tr, X_val)
        oof[val_idx] = val_pred
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
    overall_auc = roc_auc_score(y_ser, oof)
    elapsed = time.time() - start
    print(
        f"{name:40s} OOF AUC={overall_auc:.5f}  "
        f"fold std={np.std(fold_aucs):.5f}  ({elapsed:.0f}s)"
    )
    results.append({
        "name": name,
        "oof_auc": overall_auc,
        "fold_auc_mean": np.mean(fold_aucs),
        "fold_auc_std": np.std(fold_aucs),
        "fold_aucs": fold_aucs,
        "runtime_s": elapsed,
    })
    oof_store[name] = oof
    return oof

In [ ]:
# `run_cv` builds a fresh StratifiedKFold(shuffle=True, random_state=SEED)
# on every call. Prove that's deterministic -- not just assumed -- so every
# candidate's OOF array is verified aligned to the same folds and row order
# before any cross-candidate comparison (paired bootstrap, correlation).
_fold_check_a = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
_fold_check_b = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
assert all(
    np.array_equal(a_tr, b_tr) and np.array_equal(a_val, b_val)
    for (a_tr, a_val), (b_tr, b_val) in zip(_fold_check_a, _fold_check_b)
), "StratifiedKFold splits are not deterministic across calls"
print("Fold determinism verified: identical splits across independent calls.")

### Model Factory

One factory both the evaluation and submission paths call, so the fitted submission model's estimator configuration (class and hyperparameters) is guaranteed identical to the one the OOF score below was measured on.

In [ ]:
def build_model(name: str):
    """Build a configured model without fitting it."""
    if name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=SEED,
            max_iter=200,
            categorical_features="from_dtype",
        )
    if name == "lightgbm_tuned":
        # E01 recommended candidate (docs/9_experiment_ledger.md):
        # LGBM_CONFIGS[2] (c3) from Section 9's search. Reads the same
        # config dict the search used -- single source of truth, not a
        # duplicated literal -- so a future edit to that config can't
        # silently drift out of sync with the promoted model.
        return LGBMClassifier(random_state=SEED, verbose=-1, **LGBM_CONFIGS[2])
    raise ValueError(f"Unknown model: {name}")


# Fit-time keyword arguments a model needs beyond X_train/y_train, keyed by
# build_model() name. LightGBM's categorical_feature must be passed at fit
# time (unlike HGB's constructor-level categorical_features="from_dtype"),
# and it must be identical whether the model is being evaluated in the
# Section 9 search or fit for submission -- centralized here so both call
# sites share one definition instead of two independently-written calls
# that happen to currently agree.
LIGHTGBM_FIT_KWARGS = {"categorical_feature": CATEGORICAL_FEATURES}
MODEL_FIT_KWARGS = {
    "hist_gradient_boosting": {},
    "lightgbm_tuned": LIGHTGBM_FIT_KWARGS,
}


def fit_model(name: str, X_tr: pd.DataFrame, y_tr: pd.Series):
    """Build and fit a named model, applying its centralized fit kwargs."""
    model = build_model(name)
    model.fit(X_tr, y_tr, **MODEL_FIT_KWARGS.get(name, {}))
    return model

## 4. v1 — Sanity Baselines

Constant predictor, logistic regression, and `HistGradientBoostingClassifier` establish the floor and confirm the evaluation pipeline before any tuning. Logistic regression is attempted with a stronger regularization/solver configuration; if it is still numerically unstable, it is retired (`RUN_LOGISTIC = False`) rather than reported as a measured baseline.

In [ ]:
if RUN_V1_SANITY:
    # Constant predictor: no ranking signal by construction, AUC = 0.5
    # (not computed via roc_auc_score, which requires score variation);
    # recorded directly as the theoretical floor.
    results.append({
        "name": "v1a_constant", "oof_auc": 0.5,
        "fold_auc_mean": 0.5, "fold_auc_std": 0.0, "fold_aucs": [0.5] * N_FOLDS,
    })
    print(f"{'v1a_constant':40s} OOF AUC=0.50000  (theoretical floor, not fit)")

In [ ]:
if RUN_V1_SANITY and RUN_LOGISTIC:
    def fit_predict_logreg(X_tr, y_tr, X_val):
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]), NUMERIC_FEATURES),
                ("cat", Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]), CATEGORICAL_FEATURES),
            ])),
            ("clf", LogisticRegression(
                solver="saga", penalty="l2", C=0.1, max_iter=2_000,
                random_state=SEED, n_jobs=-1,
            )),
        ])
        pipe.fit(X_tr, y_tr)
        return pipe.predict_proba(X_val)[:, 1]

    _ = run_cv("v1b_logistic_regression", fit_predict_logreg, X, y)

In [ ]:
if RUN_V1_SANITY:
    def fit_predict_hgb(X_tr, y_tr, X_val):
        # Fixed to the untuned HGB sanity floor regardless of CHAMPION_NAME
        # -- this is E01's comparison baseline (docs/9_experiment_ledger.md),
        # not "whichever model is currently promoted."
        model = fit_model("hist_gradient_boosting", X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v1c_hist_gradient_boosting", fit_predict_hgb, X, y)

**Insight:** the constant predictor's AUC=0.5 confirms the floor; HGB's actual OOF AUC (see the summary table in Section 12) confirms the pipeline is producing genuine ranking signal well above that floor before any tuning. Logistic regression is retired here (see above) rather than included in the comparison.

## 5. v2 — Strong Models (Native Categorical)

In [ ]:
if RUN_V2_STRONG:
    def fit_predict_lgbm(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    lgbm_oof = run_cv("v2a_lightgbm_native_cat", fit_predict_lgbm, X, y)

In [ ]:
if RUN_V2_STRONG:
    def catboost_ready(df: pd.DataFrame) -> pd.DataFrame:
        # CatBoost's pandas-Categorical cat_features path rejects NaN
        # directly ("cat_features must be integer or string ... NaN
        # values should be converted to string") -- give it an explicit
        # "missing" string category instead, still distinct from every
        # real level, so this is native-missing-handling in spirit even
        # though LightGBM/HGB can take the NaN itself.
        out = df.copy()
        for col in CATEGORICAL_FEATURES:
            out[col] = out[col].astype("object").fillna("missing").astype(str)
        return out

    def fit_predict_catboost(X_tr, y_tr, X_val):
        model = CatBoostClassifier(
            random_seed=SEED, iterations=200, depth=6, learning_rate=0.05,
            cat_features=CATEGORICAL_FEATURES, verbose=False,
        )
        model.fit(catboost_ready(X_tr), y_tr)
        return model.predict_proba(catboost_ready(X_val))[:, 1]

    catboost_oof = run_cv("v2b_catboost_native_cat", fit_predict_catboost, X, y)

**Insight:** compare against v1's sanity baselines — native categorical + native missing-value handling should clear the HGB floor if the tree ensembles are extracting more signal than a single boosting pass on ordinal-ish encodings.

## 6. `_is_missing` Indicator Flags — OOF Ablation

The marginal analysis during EDA found no strong target signal in missingness, but explicitly did not rule out a conditional effect. This is the actual test — LightGBM with vs. without explicit `_is_missing` columns alongside native NaN handling, same model/fold setup as v2a for a clean comparison.

In [ ]:
if RUN_V2_MISSING_ABLATION:
    X_with_flags = X.copy()
    for col in ALL_FEATURES:
        X_with_flags[f"{col}_is_missing"] = X[col].isna().astype(int)

    def fit_predict_lgbm_flags(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2c_lightgbm_plus_missing_flags", fit_predict_lgbm_flags, X_with_flags, y)

**Insight:** compare `v2a_lightgbm_native_cat` vs. `v2c_lightgbm_plus_missing_flags` OOF AUC — this is the direct answer to whether `_is_missing` flags earn their place, not the marginal EDA table.

## 7. v3 — Engineered Features

EDA-informed ratio/residual features among the three strongest predictors, computed identically on train and test, target-free (no leakage from the target into feature construction):

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add EDA-informed ratio/residual features. Target-free, safe to
    compute once outside the CV loop."""
    out = df.copy()
    out["social_to_screen_ratio"] = df["social_media_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["gaming_to_screen_ratio"] = df["gaming_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["time_budget_residual"] = 24 - (
        df["sleep_hours"] + df["work_study_hours"] + df["daily_screen_time_hours"]
    )
    out["weekend_escalation"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    return out

ENGINEERED_FEATURES = [
    "social_to_screen_ratio", "gaming_to_screen_ratio",
    "time_budget_residual", "weekend_escalation",
]

if RUN_V3_ENGINEERED:
    X_engineered = add_engineered_features(X)

    def fit_predict_lgbm_engineered(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v3_lightgbm_plus_engineered", fit_predict_lgbm_engineered, X_engineered, y)

**Insight:** compare `v3_lightgbm_plus_engineered` against `v2a_lightgbm_native_cat` — ratios/residuals of features a tree ensemble can already split on nonlinearly are not guaranteed to help.

## 8. Class-Imbalance A/B

AUC is rank-based, so `class_weight` mainly affects optimizer dynamics rather than the final ranking — tested directly rather than assumed either way.

In [ ]:
if RUN_CLASS_WEIGHT_ABLATION:
    def fit_predict_lgbm_balanced(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1, class_weight="balanced",
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2d_lightgbm_class_weight_balanced", fit_predict_lgbm_balanced, X, y)

**Insight:** compare against `v2a_lightgbm_native_cat` (unweighted) — expect little to no AUC change either way since AUC only depends on score ranking.

## 9. Comparable Hand-Designed Tuning (E01)

Tests whether LightGBM or CatBoost can close the untuned-HGB gap from Section 4 when given comparable learning-rate/iteration budgets — and gives HGB its own tuned configurations too, so this is a genuine three-family search rather than two families chasing a frozen number. Up to four hand-designed configurations per family, no automated search. Every configuration and result is recorded in `docs/9_experiment_ledger.md`, including configurations that do not clear the gate; clearing the gate produces a recommendation for promotion, not an automatic champion change.

In [ ]:
HGB_CONFIGS = [
    {"max_iter": 400, "learning_rate": 0.05, "max_leaf_nodes": 31},
    {"max_iter": 600, "learning_rate": 0.03, "max_leaf_nodes": 31},
    {"max_iter": 400, "learning_rate": 0.05, "max_leaf_nodes": 63},
    {"max_iter": 600, "learning_rate": 0.03, "max_leaf_nodes": 63},
]

LGBM_CONFIGS = [
    {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 31},
    {"n_estimators": 600, "learning_rate": 0.03, "num_leaves": 31},
    {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63},
    {"n_estimators": 600, "learning_rate": 0.03, "num_leaves": 63},
]

CATBOOST_CONFIGS = [
    {"iterations": 400, "learning_rate": 0.05, "depth": 6},
    {"iterations": 600, "learning_rate": 0.03, "depth": 6},
    {"iterations": 400, "learning_rate": 0.05, "depth": 8},
    {"iterations": 600, "learning_rate": 0.03, "depth": 8},
]


def paired_auc_bootstrap(
    y_true: np.ndarray,
    champion_pred: np.ndarray,
    candidate_pred: np.ndarray,
    n_bootstrap: int = 200,
    bootstrap_size: int = 100_000,
    seed: int = 42,
) -> dict:
    """Estimate paired AUC-difference uncertainty without storing samples."""
    rng = np.random.default_rng(seed)
    positive = np.flatnonzero(y_true == 1)
    negative = np.flatnonzero(y_true == 0)
    positive_size = round(bootstrap_size * len(positive) / len(y_true))
    negative_size = bootstrap_size - positive_size
    differences = np.empty(n_bootstrap)
    for index in range(n_bootstrap):
        sample = np.concatenate(
            [
                rng.choice(positive, positive_size, replace=True),
                rng.choice(negative, negative_size, replace=True),
            ]
        )
        differences[index] = (
            roc_auc_score(y_true[sample], candidate_pred[sample])
            - roc_auc_score(y_true[sample], champion_pred[sample])
        )
    return {
        "mean_delta": float(differences.mean()),
        "lower_95": float(np.quantile(differences, 0.025)),
        "upper_95": float(np.quantile(differences, 0.975)),
        "probability_positive": float((differences > 0).mean()),
    }

In [ ]:
if RUN_E01_TUNING:
    def fit_predict_hgb_config(config):
        def fit_predict(X_tr, y_tr, X_val):
            model = HistGradientBoostingClassifier(
                random_state=SEED, categorical_features="from_dtype", **config,
            )
            model.fit(X_tr, y_tr)
            return model.predict_proba(X_val)[:, 1]
        return fit_predict

    for i, config in enumerate(HGB_CONFIGS, start=1):
        _ = run_cv(f"e01_hgb_c{i}", fit_predict_hgb_config(config), X, y)

    def fit_predict_lgbm_config(config):
        def fit_predict(X_tr, y_tr, X_val):
            model = LGBMClassifier(random_state=SEED, verbose=-1, **config)
            # Same LIGHTGBM_FIT_KWARGS the submission path uses via
            # fit_model() -- one definition, not two independently-written
            # calls that happen to currently agree.
            model.fit(X_tr, y_tr, **LIGHTGBM_FIT_KWARGS)
            return model.predict_proba(X_val)[:, 1]
        return fit_predict

    for i, config in enumerate(LGBM_CONFIGS, start=1):
        _ = run_cv(f"e01_lightgbm_c{i}", fit_predict_lgbm_config(config), X, y)

    def fit_predict_catboost_config(config):
        def fit_predict(X_tr, y_tr, X_val):
            model = CatBoostClassifier(
                random_seed=SEED, cat_features=CATEGORICAL_FEATURES,
                verbose=False, **config,
            )
            model.fit(catboost_ready(X_tr), y_tr)
            return model.predict_proba(catboost_ready(X_val))[:, 1]
        return fit_predict

    for i, config in enumerate(CATBOOST_CONFIGS, start=1):
        _ = run_cv(f"e01_catboost_c{i}", fit_predict_catboost_config(config), X, y)

In [ ]:
if RUN_E01_TUNING:
    E01_CANDIDATES = (
        [f"e01_hgb_c{i}" for i in range(1, len(HGB_CONFIGS) + 1)]
        + [f"e01_lightgbm_c{i}" for i in range(1, len(LGBM_CONFIGS) + 1)]
        + [f"e01_catboost_c{i}" for i in range(1, len(CATBOOST_CONFIGS) + 1)]
    )
    CHAMPION_RESULT_NAME = "v1c_hist_gradient_boosting"
    y_np = y.to_numpy()
    champion_oof = oof_store[CHAMPION_RESULT_NAME]
    champion_result = next(r for r in results if r["name"] == CHAMPION_RESULT_NAME)

    e01_rows = []
    for name in E01_CANDIDATES:
        candidate_oof = oof_store[name]
        candidate_result = next(r for r in results if r["name"] == name)
        bootstrap = paired_auc_bootstrap(y_np, champion_oof, candidate_oof)
        correlation = float(np.corrcoef(champion_oof, candidate_oof)[0, 1])
        folds_beaten = int(sum(
            c > h for c, h in zip(candidate_result["fold_aucs"], champion_result["fold_aucs"])
        ))
        e01_rows.append({
            "name": name,
            "oof_auc": candidate_result["oof_auc"],
            "fold_aucs": candidate_result["fold_aucs"],
            "fold_auc_std": candidate_result["fold_auc_std"],
            "folds_beaten": folds_beaten,
            "runtime_s": candidate_result["runtime_s"],
            "paired_mean_delta": bootstrap["mean_delta"],
            "paired_lower_95": bootstrap["lower_95"],
            "paired_upper_95": bootstrap["upper_95"],
            "probability_positive": bootstrap["probability_positive"],
            "correlation_with_champion": correlation,
        })

    e01_table = pd.DataFrame(e01_rows).sort_values(
        "paired_mean_delta", ascending=False
    ).reset_index(drop=True)
    display(e01_table)
    # display() output is not retrievable from a Kaggle kernel's execution
    # log (only print() is) -- print the same table, plus each candidate's
    # exact 5 fold AUCs individually (the table's own column is a list and
    # doesn't print in full via to_string()), so this run's evidence is
    # self-contained without needing __notebook__.ipynb.
    print(e01_table.drop(columns=["fold_aucs"]).to_string(index=False))
    for row in e01_rows:
        print(f"{row['name']:20s} fold_aucs={[round(a, 5) for a in row['fold_aucs']]}")

In [ ]:
if RUN_E01_TUNING:
    # Predeclared in docs/9_experiment_ledger.md before this cell was run:
    # a candidate clears the gate only if it beats the champion on a
    # majority of folds (>=3 of 5) AND the paired 95% interval is entirely
    # positive AND the bootstrap's probability of a positive delta is
    # >=0.95. All three conditions are required; every candidate's result
    # is kept regardless. Clearing the gate produces a *recommendation*
    # only -- CHAMPION_NAME is not changed here. Promoting a recommended
    # candidate to champion is a separate, explicit decision, not an automatic
    # consequence of this cell.
    def passes_e01_promotion_gate(row: dict) -> bool:
        return (
            row["folds_beaten"] >= 3
            and row["paired_lower_95"] > 0.0
            and row["probability_positive"] >= 0.95
        )

    # The candidate documented in docs/9_experiment_ledger.md and wired
    # into build_model("lightgbm_tuned") as the recommendation. If the gate
    # winner ever disagrees, that's a documentation/code drift bug -- fail
    # loudly rather than silently recommending something build_model()
    # doesn't actually implement.
    E01_RECOMMENDED_CANDIDATE = "e01_lightgbm_c3"

    e01_gate_cleared = [row for row in e01_rows if passes_e01_promotion_gate(row)]
    if e01_gate_cleared:
        E01_WINNER = max(e01_gate_cleared, key=lambda row: row["paired_mean_delta"])["name"]
        assert E01_WINNER == E01_RECOMMENDED_CANDIDATE, (
            f"Gate winner {E01_WINNER!r} does not match the documented "
            f"recommendation {E01_RECOMMENDED_CANDIDATE!r} -- update "
            "docs/9_experiment_ledger.md and build_model('lightgbm_tuned') "
            "before treating this as the recommendation."
        )
        print(
            f"E01 recommendation: {E01_WINNER} clears the gate "
            f"(approved working champion: CHAMPION_NAME={CHAMPION_NAME!r})."
        )
    else:
        E01_WINNER = None
        print(
            "E01 recommendation: no candidate clears the predeclared gate; "
            f"retaining {CHAMPION_RESULT_NAME}."
        )

**Insight:** the recommendation gate was fixed before this cell ran (majority of folds beaten, entire paired 95% interval positive, probability of a positive delta at least 0.95) so the result above is not chosen after seeing which threshold flatters it. Clearing the gate produces a recommendation for the user's promotion decision, not an automatic champion change. `docs/9_experiment_ledger.md` records every configuration's exact numbers, including rejected ones, along with the caveats that limit how strong a claim this recommendation supports.

## 10. Model Diversity Check: Working Champion Vs. Tuned HGB

Tests whether the working champion (`lightgbm_tuned`) and its closest challenger from Section 9 (the tuned HGB configuration, refit here as `e02_hgb_c3`) make different enough predictions to be worth combining. Both models are refit under new names to get fresh OOF arrays for this section and as an independent reproducibility check on Section 9's numbers. The correlation threshold, the complementary-residual criteria (decile disagreement, 0.5-threshold error-set Jaccard overlap), and the decision rule below are all predeclared in `docs/9_experiment_ledger.md` before this section was run.

In [ ]:
if RUN_E02_ENTRY_CHECK:
    def fit_predict_e02_lightgbm(X_tr, y_tr, X_val):
        # Route through the shared champion factory/fit path (not a manual
        # reconstruction), and assert the expected identity, so a future
        # CHAMPION_NAME or build_model() change can't silently make this
        # comparison stale.
        assert CHAMPION_NAME == "lightgbm_tuned", (
            f"This check assumes the champion is 'lightgbm_tuned', but "
            f"CHAMPION_NAME={CHAMPION_NAME!r} -- update "
            "fit_predict_e02_lightgbm before trusting this comparison."
        )
        model = fit_model("lightgbm_tuned", X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    e02_lightgbm_oof = run_cv("e02_lightgbm_tuned", fit_predict_e02_lightgbm, X, y)

    def fit_predict_e02_hgb(X_tr, y_tr, X_val):
        model = HistGradientBoostingClassifier(
            random_state=SEED, categorical_features="from_dtype", **HGB_CONFIGS[2],
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    e02_hgb_oof = run_cv("e02_hgb_c3", fit_predict_e02_hgb, X, y)

In [ ]:
if RUN_E02_ENTRY_CHECK:
    e02_lightgbm_result = next(r for r in results if r["name"] == "e02_lightgbm_tuned")
    e02_hgb_result = next(r for r in results if r["name"] == "e02_hgb_c3")

    # Fold-by-fold deltas (lightgbm - hgb), per fold.
    e02_fold_deltas = [
        lgbm_auc - hgb_auc
        for lgbm_auc, hgb_auc in zip(e02_lightgbm_result["fold_aucs"], e02_hgb_result["fold_aucs"])
    ]
    print(f"e02_lightgbm_tuned fold_aucs={[round(a, 5) for a in e02_lightgbm_result['fold_aucs']]}")
    print(f"e02_hgb_c3         fold_aucs={[round(a, 5) for a in e02_hgb_result['fold_aucs']]}")
    print(f"fold-by-fold delta (lightgbm - hgb)={[round(d, 5) for d in e02_fold_deltas]}")

    # Direct paired bootstrap between the two candidates themselves (not
    # either vs. the original v1c champion) -- quantifies uncertainty in
    # their own ~0.00027 gap directly. Conditions on these fixed OOF fits:
    # it does not cover fold/seed/refit uncertainty, this pair was selected
    # as the two strongest Section 9 configurations evaluated on the same
    # OOF data used to select them, and it is not a multiplicity-corrected
    # test (Section 9's results discuss these caveats more fully).
    y_np = y.to_numpy()
    e02_bootstrap = paired_auc_bootstrap(y_np, e02_hgb_oof, e02_lightgbm_oof)
    print(
        f"Direct paired bootstrap (lightgbm vs hgb): mean_delta={e02_bootstrap['mean_delta']:.6f} "
        f"95%=[{e02_bootstrap['lower_95']:.6f}, {e02_bootstrap['upper_95']:.6f}] "
        f"resamples_positive={int(e02_bootstrap['probability_positive'] * 200)}/200"
    )

    # Correlation: Pearson (the metric the 0.995 threshold applies to) and
    # Spearman (rank correlation, for context).
    e02_pearson = float(np.corrcoef(e02_lightgbm_oof, e02_hgb_oof)[0, 1])
    e02_spearman = float(pd.Series(e02_lightgbm_oof).corr(pd.Series(e02_hgb_oof), method="spearman"))
    print(f"Pearson correlation={e02_pearson:.6f}  Spearman correlation={e02_spearman:.6f}")

    # Decile ranking disagreement: top/bottom 10% by predicted probability.
    n_rows = len(e02_lightgbm_oof)
    decile_size = n_rows // 10
    lgbm_order = np.argsort(e02_lightgbm_oof)
    hgb_order = np.argsort(e02_hgb_oof)
    lgbm_top, lgbm_bottom = set(lgbm_order[-decile_size:]), set(lgbm_order[:decile_size])
    hgb_top, hgb_bottom = set(hgb_order[-decile_size:]), set(hgb_order[:decile_size])
    top_overlap = len(lgbm_top & hgb_top) / decile_size
    bottom_overlap = len(lgbm_bottom & hgb_bottom) / decile_size
    print(
        f"Top-decile overlap={top_overlap:.4f} (disagreement={1 - top_overlap:.4f})  "
        f"Bottom-decile overlap={bottom_overlap:.4f} (disagreement={1 - bottom_overlap:.4f})"
    )

    # Error-set overlap at a 0.5 threshold -- the predeclared complementary-
    # residual metric. Jaccard = both wrong / union of both models' wrong sets.
    lgbm_wrong = (e02_lightgbm_oof >= 0.5).astype(int) != y_np
    hgb_wrong = (e02_hgb_oof >= 0.5).astype(int) != y_np
    both_wrong = int((lgbm_wrong & hgb_wrong).sum())
    only_lgbm_wrong = int((lgbm_wrong & ~hgb_wrong).sum())
    only_hgb_wrong = int((~lgbm_wrong & hgb_wrong).sum())
    union_wrong = both_wrong + only_lgbm_wrong + only_hgb_wrong
    both_right = n_rows - union_wrong
    e02_error_jaccard = both_wrong / union_wrong if union_wrong else float("nan")
    print(
        f"Errors: lightgbm_total={int(lgbm_wrong.sum())} hgb_total={int(hgb_wrong.sum())} "
        f"both_wrong={both_wrong} only_lightgbm_wrong={only_lgbm_wrong} "
        f"only_hgb_wrong={only_hgb_wrong} both_right={both_right} "
        f"error_jaccard={e02_error_jaccard:.4f}"
    )

    # Predeclared decision rule (docs/9_experiment_ledger.md): proceed only
    # if Pearson correlation < 0.995 AND error-set Jaccard <= 0.90.
    E02_PROCEED = bool(e02_pearson < 0.995 and e02_error_jaccard <= 0.90)
    print(
        f"Model diversity check: {'PROCEED' if E02_PROCEED else 'SKIP'} "
        f"(pearson={e02_pearson:.6f} < 0.995: {e02_pearson < 0.995}; "
        f"error_jaccard={e02_error_jaccard:.4f} <= 0.90: {e02_error_jaccard <= 0.90})"
    )
    if E02_PROCEED:
        raise NotImplementedError(
            "The diversity check returned PROCEED, but the follow-up "
            "evaluation (an additional model family and a blend-weight "
            "sweep) is not yet implemented in this notebook. Implement it "
            "before treating this run as a completed PROCEED decision -- "
            "a rerun must not finish 'successfully' having silently done "
            "only half the required work."
        )

**Insight:** the decision above uses the exact predeclared rule from `docs/9_experiment_ledger.md` (Pearson correlation `< 0.995` and error-set Jaccard overlap `<= 0.90`), computed after both thresholds were fixed, not chosen to fit the result. A few limitations apply: the 200-resample bootstrap conditions on these fixed OOF fits and does not cover fold, seed, or refit uncertainty; this pair was selected as the two strongest configurations from Section 9's search, evaluated on the same out-of-fold data used to select them; and the reported interval is not an independent confirmation or a multiplicity-corrected test (Section 9's results discuss these caveats more fully). A `SKIP` decision reflects that these two models are not different enough by the predeclared bar to justify a combination pass within this notebook's time budget — not a claim that no combination of their predictions could ever improve on the champion alone.

## 11. Post-Launch Exploration: Feature Representation (E03)

Tests whether a different feature representation — constraint-based imputation for the screen-time group, frequency encoding, and leak-free per-exact-value target encoding, layered on the exact champion hyperparameters (`LGBM_CONFIGS[2]`) so only the features change — improves on the published champion. Three additive configurations isolate which specific piece of the representation change, if any, is responsible for a difference. The imputation constraint (`daily_screen_time_hours >= social_media_hours + gaming_hours + work_study_hours`, zero violations across all complete rows in this dataset) and the promotion gate are predeclared in `docs/9_experiment_ledger.md` before this section was run.

In [ ]:
if RUN_E03_EXPLORATION:
    def add_constrained_imputation(df: pd.DataFrame) -> pd.DataFrame:
        # Verified against data/train.csv and data/test.csv: zero
        # violations of daily_screen_time_hours >= social_media_hours +
        # gaming_hours + work_study_hours across every row where all four
        # columns are present (docs/9_experiment_ledger.md E03 section).
        # A missing member of this group is therefore bounded by the
        # others, not merely unknown.
        d = df.copy()
        comp = ["social_media_hours", "gaming_hours", "work_study_hours"]
        daily = df["daily_screen_time_hours"]
        n_missing_comp = df[comp].isna().sum(axis=1)
        daily_missing = daily.isna()
        comp_sum = df[comp].sum(axis=1, min_count=1)
        slack = daily - comp_sum
        case_daily_missing = daily_missing & (n_missing_comp == 0)
        case_one_comp_missing = (~daily_missing) & (n_missing_comp == 1)
        d["ci_daily_lb"] = np.where(case_daily_missing, comp_sum, np.nan)
        d["ci_missing_comp_ub"] = np.where(case_one_comp_missing, slack, np.nan)
        d["ci_missing_comp_mid"] = np.where(
            case_one_comp_missing, np.clip(slack / 2, 0, None), np.nan
        )
        d["ci_case"] = np.select(
            [case_daily_missing, case_one_comp_missing], [1.0, 2.0], default=0.0
        )
        d["ci_n_missing"] = (daily_missing.astype(int) + n_missing_comp).astype(float)
        return d

    def add_e03_ratios(df: pd.DataFrame) -> pd.DataFrame:
        """Rate features not already in ENGINEERED_FEATURES. Target-free."""
        d = df.copy()
        ds = df["daily_screen_time_hours"]
        sl = df["sleep_hours"]
        ws = df["weekend_screen_time"]
        ao = df["app_opens_per_day"]
        nt = df["notifications_per_day"]
        eps = 1e-5
        d["missing_count"] = df[ALL_FEATURES].isna().sum(axis=1).astype(float)
        d["screen_to_sleep_ratio"] = ds / (sl + eps)
        d["weekend_vs_daily_ratio"] = ws / (ds + eps)
        d["app_opens_per_hour"] = ao / (ds + eps)
        d["notifications_per_hour"] = nt / (ds + eps)
        return d

    X_e03 = add_e03_ratios(add_constrained_imputation(X))
    for col in CATEGORICAL_FEATURES:
        X_e03[col] = X_e03[col].astype("category")

    E03_CI_RATIO_COLUMNS = [
        "ci_daily_lb", "ci_missing_comp_ub", "ci_missing_comp_mid",
        "ci_case", "ci_n_missing", "missing_count", "screen_to_sleep_ratio",
        "weekend_vs_daily_ratio", "app_opens_per_hour", "notifications_per_hour",
    ]
    print(
        f"E03 base+CI+ratio feature count: {X_e03.shape[1]} "
        f"(base {len(ALL_FEATURES)} + {len(E03_CI_RATIO_COLUMNS)} new)"
    )

In [ ]:
if RUN_E03_EXPLORATION:
    # Frequency of each exact raw value across train+test combined --
    # target-free (feature distribution only), so leak-free by
    # construction. The string view (Xs_e03) is also what target encoding
    # below encodes on, treating every raw column -- numeric or
    # categorical -- as a set of exact discrete values.
    Xs_e03 = train[ALL_FEATURES].astype(str).fillna("na")
    Xs_test_e03 = test[ALL_FEATURES].astype(str).fillna("na")

    E03_FREQ_COLUMNS = [f"{c}_freq" for c in ALL_FEATURES]
    for col in ALL_FEATURES:
        freq = pd.concat([Xs_e03[col], Xs_test_e03[col]]).value_counts()
        X_e03[f"{col}_freq"] = Xs_e03[col].map(freq).astype(float)

    E03_TE_COLUMNS = [f"te_{c}" for c in ALL_FEATURES]
    print(f"E03 +frequency feature count: {X_e03.shape[1]}")

In [ ]:
if RUN_E03_EXPLORATION:
    def make_e03_fit_predict(use_freq: bool, use_te: bool):
        cols = list(ALL_FEATURES) + E03_CI_RATIO_COLUMNS + (
            E03_FREQ_COLUMNS if use_freq else []
        )

        def fit_predict(X_tr, y_tr, X_val):
            A_tr = X_tr[cols].copy()
            A_val = X_val[cols].copy()
            if use_te:
                # Per-exact-value target encoding, fit inside this outer
                # fold's training rows only, with the encoder's own
                # internal cv=5 cross-fit -- double cross-validation, so a
                # training row's own target never leaks into its encoded
                # value. Test rows are not encoded here (this section only
                # evaluates OOF comparison, not a submission artifact).
                encoder = TargetEncoder(
                    cv=5, smooth="auto", shuffle=True, random_state=SEED,
                    target_type="binary",
                )
                Z_tr = encoder.fit_transform(Xs_e03.loc[X_tr.index], y_tr.to_numpy())
                Z_val = encoder.transform(Xs_e03.loc[X_val.index])
                A_tr[E03_TE_COLUMNS] = Z_tr
                A_val[E03_TE_COLUMNS] = Z_val
            # Champion hyperparameters, unchanged -- isolates the feature
            # effect from a second round of tuning.
            model = LGBMClassifier(random_state=SEED, verbose=-1, **LGBM_CONFIGS[2])
            model.fit(A_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
            return model.predict_proba(A_val)[:, 1]
        return fit_predict

    _ = run_cv(
        "e03_constrained_imputation",
        make_e03_fit_predict(use_freq=False, use_te=False),
        X_e03, y,
    )
    _ = run_cv(
        "e03_plus_frequency",
        make_e03_fit_predict(use_freq=True, use_te=False),
        X_e03, y,
    )
    _ = run_cv(
        "e03_plus_target_encoding",
        make_e03_fit_predict(use_freq=True, use_te=True),
        X_e03, y,
    )

In [ ]:
if RUN_E03_EXPLORATION:
    E03_CANDIDATES = [
        "e03_constrained_imputation", "e03_plus_frequency", "e03_plus_target_encoding",
    ]
    E03_CHAMPION_RESULT_NAME = "e01_lightgbm_c3"
    y_np = y.to_numpy()
    e03_champion_oof = oof_store[E03_CHAMPION_RESULT_NAME]
    e03_champion_result = next(r for r in results if r["name"] == E03_CHAMPION_RESULT_NAME)

    e03_rows = []
    for name in E03_CANDIDATES:
        candidate_oof = oof_store[name]
        candidate_result = next(r for r in results if r["name"] == name)
        bootstrap = paired_auc_bootstrap(y_np, e03_champion_oof, candidate_oof)
        correlation = float(np.corrcoef(e03_champion_oof, candidate_oof)[0, 1])
        folds_beaten = int(sum(
            c > h for c, h in zip(candidate_result["fold_aucs"], e03_champion_result["fold_aucs"])
        ))
        e03_rows.append({
            "name": name,
            "oof_auc": candidate_result["oof_auc"],
            "fold_aucs": candidate_result["fold_aucs"],
            "fold_auc_std": candidate_result["fold_auc_std"],
            "folds_beaten": folds_beaten,
            "runtime_s": candidate_result["runtime_s"],
            "paired_mean_delta": bootstrap["mean_delta"],
            "paired_lower_95": bootstrap["lower_95"],
            "paired_upper_95": bootstrap["upper_95"],
            "probability_positive": bootstrap["probability_positive"],
            "correlation_with_champion": correlation,
        })

    e03_table = pd.DataFrame(e03_rows).sort_values(
        "paired_mean_delta", ascending=False
    ).reset_index(drop=True)
    display(e03_table.drop(columns=["fold_aucs"]))
    # print() fallback for Kaggle log capture, matching E01/E02.
    print(e03_table.drop(columns=["fold_aucs"]).to_string(index=False))
    for row in e03_rows:
        print(f"{row['name']:28s} fold_aucs={[round(a, 5) for a in row['fold_aucs']]}")

In [ ]:
if RUN_E03_EXPLORATION:
    # Predeclared in docs/9_experiment_ledger.md before this section was
    # run: majority of folds beaten (>=3/5), entirely positive paired 95%
    # interval, probability of a positive delta >=0.95 -- same structure
    # as E01's gate. This produces a recommendation only; it does not
    # change CHAMPION_NAME.
    def passes_e03_promotion_gate(row: dict) -> bool:
        return (
            row["folds_beaten"] >= 3
            and row["paired_lower_95"] > 0.0
            and row["probability_positive"] >= 0.95
        )

    e03_gate_cleared = [row for row in e03_rows if passes_e03_promotion_gate(row)]
    if e03_gate_cleared:
        E03_WINNER = max(e03_gate_cleared, key=lambda row: row["paired_mean_delta"])["name"]
        print(
            f"E03 recommendation: {E03_WINNER} clears the gate against "
            f"{E03_CHAMPION_RESULT_NAME} (not promoted; CHAMPION_NAME "
            f"remains {CHAMPION_NAME!r})."
        )
    else:
        E03_WINNER = None
        print(
            "E03 recommendation: no configuration clears the predeclared "
            f"gate; retaining {CHAMPION_NAME!r}."
        )

    for row in e03_rows:
        cleared = passes_e03_promotion_gate(row)
        print(
            f"{row['name']:28s} gate={'CLEAR' if cleared else 'fail'} "
            f"folds_beaten={row['folds_beaten']}/5 "
            f"lower_95={row['paired_lower_95']:.6f} "
            f"prob_positive={row['probability_positive']:.4f}"
        )

**Insight:** the three configurations are additive by design, so a jump between two of them attributes the gain to a specific technique (imputation, frequency encoding, or target encoding) rather than a single opaque "new features helped" result. All three still use the champion's exact hyperparameters — no new tuning is layered on top, so any improvement is a feature-representation effect, not a confound with a second search. The same caveats as E01/E02 apply: three configurations were compared against the same OOF data used to pick the best of them (selection bias), and the paired bootstrap does not cover fold/seed/refit uncertainty. `docs/9_experiment_ledger.md` records the full results and, if any configuration clears the gate, the recommendation is a candidate for a future promotion decision through the same review process as `lightgbm_tuned` went through — not an automatic champion change.

## 12. Summary and Candidate Sanity Checks

In [ ]:
if RUN_SUMMARY:
    summary = pd.DataFrame(results)[["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]]
    summary = summary.sort_values("oof_auc", ascending=False).reset_index(drop=True)
    display(summary)
    print(summary.to_string(index=False))

In [ ]:
def candidate_sanity_checks(name: str, oof_pred: np.ndarray, y_true: pd.Series) -> dict:
    """Sanity-check predictions without assuming a classification threshold
    AUC optimization doesn't make."""
    finite_in_range = bool(np.all(np.isfinite(oof_pred)) and np.all((oof_pred >= 0) & (oof_pred <= 1)))
    n_unique = int(pd.Series(oof_pred).nunique())
    overall_auc = roc_auc_score(y_true, oof_pred)
    return {
        "name": name,
        "finite_in_[0,1]": finite_in_range,
        "n_unique_predictions": n_unique,
        "prediction_range": (float(oof_pred.min()), float(oof_pred.max())),
        "overall_oof_auc": overall_auc,
    }

if RUN_SUMMARY:
    best_name = summary.iloc[0]["name"] if summary.iloc[0]["name"] != "v1a_constant" else summary.iloc[1]["name"]
    checks = candidate_sanity_checks(best_name, oof_store[best_name], y)
    display(pd.Series(checks))
    print(checks)

**Insight:** the leading candidate (excluding the constant floor) passes basic sanity (finite, bounded, non-degenerate predictions) before being considered for further tuning.

## 13. Next Moves

The current working champion is `lightgbm_tuned`, promoted after Section 9's comparable-budget search cleared the predeclared gate and received review approval. The untuned `v1c_hist_gradient_boosting` result remains the fixed comparison baseline; `docs/9_experiment_ledger.md` records the full evidence, caveats, and approval decision.

## 14. Submission

Builds a schema-safe submission from the champion model (`build_model(CHAMPION_NAME)`, the same factory Section 4 uses for evaluation), fit on all training rows. Only runs in submission mode (`RUN_MODE = "submission"`); evaluation mode stops above.

In [ ]:
def fit_champion_and_predict(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
) -> np.ndarray:
    """Fit the selected configuration on all training rows."""
    model = fit_model(model_name, X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


def build_submission(
    sample: pd.DataFrame,
    test_ids: pd.Series,
    predictions: np.ndarray,
) -> pd.DataFrame:
    """Create a schema-safe submission in test-row order."""
    submission = sample.copy()
    submission["id"] = test_ids.to_numpy()
    submission["addicted_label"] = predictions
    if submission.columns.tolist() != ["id", "addicted_label"]:
        raise ValueError("Unexpected submission columns")
    if not submission["id"].equals(test_ids.reset_index(drop=True)):
        raise ValueError("Submission ID order mismatch")
    if not np.isfinite(predictions).all():
        raise ValueError("Non-finite predictions")
    if not ((predictions >= 0.0) & (predictions <= 1.0)).all():
        raise ValueError("Predictions outside [0, 1]")
    return submission


if RUN_MODE == "submission":
    print(
        f"NOTEBOOK_VERSION={NOTEBOOK_VERSION} "
        f"CHAMPION_NAME={CHAMPION_NAME} SEED={SEED}"
    )
    predictions = fit_champion_and_predict(CHAMPION_NAME, X, y, X_test)
    submission = build_submission(sample, test["id"], predictions)
    OUTPUT_PATH = (
        "/kaggle/working/submission.csv"
        if os.path.exists("/kaggle/working")
        else "../submission.csv"
    )
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote {OUTPUT_PATH}: {submission.shape}")
